# ILLM-Lab11-Assignment
Probing BERT layers for sentence length, bigram shift, and subject number.

In [7]:
import os
# Must be set BEFORE torch / transformers are imported
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"   # unsupported ops fall back to CPU instead of crashing
os.environ["USE_TF"] = "0"                        # this notebook only needs PyTorch; ignore a broken TensorFlow install
os.environ["USE_FLAX"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import urllib.request
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score
from tqdm.auto import tqdm
import numpy as np

# Set random seed for reproducibility
np.random.seed(42)
torch.manual_seed(42)

In [8]:
# Define raw data URLs from the SentEval repository
base_url = "https://raw.githubusercontent.com/facebookresearch/SentEval/main/data/probing/"
datasets = {
    "sentence_length": "sentence_length.txt",
    "bigram_shift": "bigram_shift.txt",
    "subj_number": "subj_number.txt"
}

# Download files if they do not exist locally
for name, filename in datasets.items():
    if not os.path.exists(filename):
        print(f"Downloading {filename}...")
        urllib.request.urlretrieve(base_url + filename, filename)
print("Data download complete.")

Data download complete.


In [9]:
def prepare_dataset(filename, total_samples=50000, test_size=0.10):
    # SentEval probing datasets are TSV format: split \t label \t sentence
    df = pd.read_csv(filename, sep='\t', header=None, names=['split', 'label', 'sentence'])
    
    # 1. Balance the dataset to total_samples (same number of samples from all classes)
    classes = df['label'].unique()
    samples_per_class = total_samples // len(classes)
    
    balanced_dfs = []
    for cls in classes:
        cls_df = df[df['label'] == cls]
        # Sample exactly samples_per_class without replacement
        sampled_cls = cls_df.sample(n=samples_per_class, random_state=42)
        balanced_dfs.append(sampled_cls)
        
    balanced_df = pd.concat(balanced_dfs).reset_index(drop=True)
    
    # 2. Split into 90:10 ratio while maintaining class balance (stratify)
    X = balanced_df['sentence'].tolist()
    y = balanced_df['label'].tolist()
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, stratify=y, random_state=42
    )
    
    return X_train, X_test, y_train, y_test

# Prepare all three datasets
tasks_data = {}
for task_name, filename in datasets.items():
    print(f"Preparing {task_name} dataset...")
    X_train, X_test, y_train, y_test = prepare_dataset(filename)
    tasks_data[task_name] = (X_train, X_test, y_train, y_test)

Preparing sentence_length dataset...
Preparing bigram_shift dataset...
Preparing subj_number dataset...


In [10]:
# bert-base-uncased with 12 encoder layers
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
# output_hidden_states=True is required to get representations from all 12 encoder layers
model = AutoModel.from_pretrained(model_name, output_hidden_states=True)

# Apple GPU (MPS) > CUDA > CPU
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

# Half precision roughly doubles speed on the Apple GPU. Features are converted back to
# float32 before the classifier. Set to False if you want exact full-precision numbers.
USE_FP16   = device.type in ("mps", "cuda")
MAX_LENGTH = 128          # SentEval sentences are short; this just caps outliers
BATCH_SIZE = 256          # lower to 128 if you run out of GPU memory

if USE_FP16:
    model.half()
model.to(device)
model.eval()
print(f"Model loaded on {device} | fp16: {USE_FP16}")

Model loaded on mps | fp16: True


In [11]:
def extract_bert_features(sentences, batch_size=BATCH_SIZE):
    """
    Extracts the [CLS] token representation for all 12 BERT layers.
    Returns {layer_num: features_matrix (n_sentences x 768)}.

    Speed-ups for the Mac GPU:
      * tokenise once, sort sentences by length so every batch has almost no padding
      * stack the 12 layer [CLS] vectors on the GPU and copy to the CPU once per batch (not 12 times)
      * inference_mode + optional fp16
    The original sentence order is restored at the end.
    """
    enc = tokenizer(sentences, truncation=True, max_length=MAX_LENGTH)["input_ids"]
    order = np.argsort([len(e) for e in enc])
    pad_id = tokenizer.pad_token_id
    out = np.zeros((len(sentences), 12, model.config.hidden_size), dtype=np.float32)

    with torch.inference_mode():
        for start in tqdm(range(0, len(order), batch_size), desc="Extracting features"):
            idx = order[start:start + batch_size]
            longest = max(len(enc[i]) for i in idx)
            input_ids = torch.full((len(idx), longest), pad_id, dtype=torch.long)
            attention_mask = torch.zeros((len(idx), longest), dtype=torch.long)
            for row, i in enumerate(idx):
                input_ids[row, :len(enc[i])] = torch.tensor(enc[i])
                attention_mask[row, :len(enc[i])] = 1

            hidden = model(input_ids=input_ids.to(device),
                           attention_mask=attention_mask.to(device)).hidden_states
            # hidden[0] = embeddings, hidden[1..12] = encoder layers -> [CLS] of each: (batch, 12, 768)
            cls = torch.stack([h[:, 0, :] for h in hidden[1:]], dim=1)
            out[idx] = cls.float().cpu().numpy()

    if device.type == "mps":
        torch.mps.empty_cache()
    return {layer: out[:, layer - 1, :] for layer in range(1, 13)}


def cached_features(task_name, split_name, sentences):
    """Extract once, then reuse from disk so a restarted kernel does not redo the GPU work."""
    path = f"features_{task_name}_{split_name}.npz"
    if os.path.exists(path):
        data = np.load(path)
        return {layer: data[str(layer)] for layer in range(1, 13)}
    feats = extract_bert_features(sentences)
    np.savez(path, **{str(k): v for k, v in feats.items()})
    return feats

In [12]:
results_summary = {}

for task_name, (X_train, X_test, y_train, y_test) in tasks_data.items():
    print(f"\nProcessing Task: {task_name}")

    print("Extracting training features...")
    train_features = cached_features(task_name, "train", X_train)
    print("Extracting testing features...")
    test_features = cached_features(task_name, "test", X_test)

    task_results = {}

    # Train and evaluate 12 classifiers (one per BERT layer)
    for layer in range(1, 13):
        # Standardising the 768 features makes the logistic regression converge much faster
        classifier = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
        classifier.fit(train_features[layer], y_train)
        y_pred = classifier.predict(test_features[layer])

        acc = accuracy_score(y_test, y_pred)
        task_results[layer] = acc
        print(f"  Layer {layer} Accuracy: {acc:.4f}")

    results_summary[task_name] = task_results

# Display the final comparison table
print("\n=== Final Results Comparison (Test Accuracy) ===")
results_df = pd.DataFrame(results_summary)
results_df.index.name = 'BERT Layer'
display(results_df)


Processing Task: sentence_length
Extracting training features...


Extracting features:   0%|          | 0/176 [00:00<?, ?it/s]

Extracting testing features...


Extracting features:   0%|          | 0/20 [00:00<?, ?it/s]

  Layer 1 Accuracy: 0.8680
  Layer 2 Accuracy: 0.8668
  Layer 3 Accuracy: 0.8786
  Layer 4 Accuracy: 0.8718
  Layer 5 Accuracy: 0.8484
  Layer 6 Accuracy: 0.8204
  Layer 7 Accuracy: 0.7656
  Layer 8 Accuracy: 0.7606
  Layer 9 Accuracy: 0.7224
  Layer 10 Accuracy: 0.7062
  Layer 11 Accuracy: 0.6702
  Layer 12 Accuracy: 0.6276

Processing Task: bigram_shift
Extracting training features...


...

Extracting features:   0%|          | 0/176 [00:00<?, ?it/s]

Extracting testing features...


Extracting features:   0%|          | 0/20 [00:00<?, ?it/s]

  Layer 1 Accuracy: 0.8202
  Layer 2 Accuracy: 0.8290
  Layer 3 Accuracy: 0.8390
  Layer 4 Accuracy: 0.8366
  Layer 5 Accuracy: 0.8644
  Layer 6 Accuracy: 0.8810
  Layer 7 Accuracy: 0.8736
  Layer 8 Accuracy: 0.8732
  Layer 9 Accuracy: 0.8692
  Layer 10 Accuracy: 0.8682
  Layer 11 Accuracy: 0.8494
  Layer 12 Accuracy: 0.8378

=== Final Results Comparison (Test Accuracy) ===


,sentence_length,bigram_shift,subj_number
BERT Layer,,,
1,0.8680,0.4998,0.8202
2,0.8668,0.5592,0.8290
3,0.8786,0.6284,0.8390
4,0.8718,0.7182,0.8366
5,0.8484,0.7874,0.8644
6,0.8204,0.8024,0.8810
7,0.7656,0.8044,0.8736
8,0.7606,0.8136,0.8732
9,0.7224,0.8420,0.8692
